**Intelligent Vision Systems · Lab 6 · Camera Calibration**
Companion to **Lecture 6 — Camera Calibration**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_6/05-charuco-and-fisheye.ipynb) &nbsp; [Lecture 6 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/06_Camera_Calibration.html#/charuco-calibration) &nbsp; [All Lab 6 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-6-calibration)

# Lab 6E · ChArUco & Fisheye Calibration with a Known Ground Truth

_Lecture 6, Section 6: ChArUco boards and fisheye lenses._

With a real camera you never know the *true* K. Here we **render** pictures of a board with a hidden virtual camera
(exact ray tracing: `C.render_board`), calibrate from the pictures, and compare the result with the truth.
First a ChArUco board with a standard lens, then a 190° fisheye.

In [ ]:
# Setup (only needed on Google Colab or a fresh environment).
# Uncomment the next line to install the packages this notebook uses:
# !pip install -q opencv-python numpy matplotlib

import os, urllib.request

# Fetch the lab files if they are not next to the notebook.
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_6"
for f in ['lab6_checks.py']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("data ready")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import lab6_checks as C
np.set_printoptions(precision=4, suppress=True)
print("OpenCV", cv2.__version__)

## 1. Make a ChArUco board

7 × 5 squares of 40 mm, markers of 30 mm, dictionary `DICT_5X5_100`. `generateImage` gives the printable picture
(and here, the texture we render). Save `board.png` and print it for Lab 6H if you like.

In [ ]:
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_100)
board = cv2.aruco.CharucoBoard((7, 5), 40.0, 30.0, aruco_dict)    # units: mm
texture = board.generateImage((1400, 1000), marginSize=0)          # 280 mm x 200 mm -> 0.2 mm per pixel
cv2.imwrite("charuco_7x5.png", texture)
plt.figure(figsize=(7, 5)); plt.imshow(texture, cmap="gray"); plt.axis("off"); plt.show()

## 2. A hidden camera takes 20 pictures

In [ ]:
K_true = np.array([[905.0, 0, 652.0], [0, 903.0, 371.0], [0, 0, 1]])
D_true = np.array([-0.21, 0.09, 0.0008, -0.0005, -0.02])
size = (1280, 720)

views = []
for k, (r, t) in enumerate(C.random_poses(20, seed=3)):
    v = C.render_board(texture, 0.2, K_true, D_true, r, t, size)
    if k % 4 == 0:                                       # a 'hand' covers part of every 4th view
        cv2.circle(v, (int(np.random.default_rng(k).uniform(300, 980)), 360), 140, 90, -1)
    views.append(cv2.GaussianBlur(v, (3, 3), 0))

fig, ax = plt.subplots(2, 4, figsize=(20, 6))
for a, v in zip(ax.ravel(), views): a.imshow(v, cmap="gray"); a.axis("off")
plt.show()

## 3. Detect and calibrate (modern API, OpenCV ≥ 4.7)

```python
detector = cv2.aruco.CharucoDetector(board)
ch_corners, ch_ids, marker_corners, marker_ids = detector.detectBoard(gray)
obj, img = board.matchImagePoints(ch_corners, ch_ids)    # 3-D board points + 2-D image points
```
Collect `obj, img` for every view with at least 6 corners, then call `cv2.calibrateCamera`.

In [ ]:
detector = cv2.aruco.CharucoDetector(board)
all_obj, all_img = [], []
for v in views:
    ch_corners, ch_ids, m_corners, m_ids = detector.detectBoard(v)
    # TODO: if enough corners were found, match them to board points and store them
    pass

print(len(all_obj), "usable views")
# TODO: calibrate
rms, K, D = 99.0, np.eye(3), np.zeros(5)
print("RMS", rms); print("K =\n", K); print("K_true =\n", K_true); print("D =", D.ravel(), "\nD_true =", D_true)

In [ ]:
C.check_charuco(K, K_true, rms)

<details><summary><b>Show a solution</b></summary>

```python
for v in views:
    ch_corners, ch_ids, m_corners, m_ids = detector.detectBoard(v)
    if ch_ids is not None and len(ch_ids) >= 6:
        obj, img = board.matchImagePoints(ch_corners, ch_ids)
        all_obj.append(obj); all_img.append(img)
rms, K, D, rvecs, tvecs = cv2.calibrateCamera(all_obj, all_img, size, None, None)
```
</details>

Note how the **partly covered** views still contribute corners: a plain chessboard would have to skip them.

## 4. A fisheye camera

A 190° lens follows the Kannala–Brandt model, $\theta_d = \theta(1 + k_1\theta^2 + k_2\theta^4 + k_3\theta^6 + k_4\theta^8)$.
We render a 9 × 6 chessboard through it and calibrate with `cv2.fisheye.calibrate`.

In [ ]:
sq = 30                                                     # mm per square
s = 5 * sq                                                  # 150 px per square at 0.2 mm per pixel; 10 x 7 squares
cb = np.full((7 * s + 200, 10 * s + 200), 255, np.uint8)
for r in range(7):
    for c in range(10):
        if (r + c) % 2 == 0: cb[100 + r * s:100 + (r + 1) * s, 100 + c * s:100 + (c + 1) * s] = 0

Kf_true = np.array([[360.0, 0, 640], [0, 360.0, 480], [0, 0, 1]])
Df_true = np.array([0.03, -0.012, 0.004, -0.0008])
fviews = []
for r, t in C.random_poses(25, seed=11, dist_mm=(260, 420), tilt_deg=25, board_mm=(cb.shape[1] * 0.2, cb.shape[0] * 0.2)):
    fviews.append(C.render_board(cb, 0.2, Kf_true, Df_true, r, t, (1280, 960), fisheye=True, background=110))
fig, ax = plt.subplots(1, 4, figsize=(22, 4.5))
for a, v in zip(ax, fviews): a.imshow(v, cmap="gray"); a.axis("off")
plt.show()

> **OpenCV 5 note:** the fisheye flags moved from `cv2.fisheye.CALIB_*` to `cv2.CALIB_*` (with new values).
> `C.fisheye_flags()` picks the right ones for your version.

`cv2.fisheye.calibrate` is strict about its inputs: **float64** arrays, object points shaped **(1, N, 3)** and image points
**(1, N, 2)** (OpenCV 5 rejects the old (N, 1, 3) layout; (1, N, 3) works in 4.x and 5.x). The inner corners are 9 × 6, and the board's first inner corner is at (100 px + 150 px) in the texture,
but object points can use any origin, so plain `C.board_points(9, 6, sq)` works.

In [ ]:
objp = C.board_points(9, 6, sq).reshape(1, -1, 3).astype(np.float64)
f_obj, f_img = [], []
for v in fviews:
    ok, c = C.find_corners(v, (9, 6))
    if ok:
        f_obj.append(objp); f_img.append(c.reshape(1, -1, 2).astype(np.float64))
print(len(f_obj), "of", len(fviews), "views usable")

# Fisheye calibration is fragile without a starting point. For an equidistant lens that sees 180 degrees across
# the image width, r = f*theta gives f ~ W / pi. Start there and let the optimiser refine it.
K0 = np.array([[1280 / np.pi, 0, 640], [0, 1280 / np.pi, 480], [0, 0, 1]])
flags = C.fisheye_flags(use_guess=True)   # cv2.fisheye.CALIB_* in OpenCV 4, cv2.CALIB_* in OpenCV 5
rms_f, Kf, Df, _, _ = cv2.fisheye.calibrate(f_obj, f_img, (1280, 960), K0, np.zeros(4), flags=flags,
                                            criteria=(cv2.TERM_CRITERIA_COUNT + cv2.TERM_CRITERIA_EPS, 100, 1e-8))
print("RMS", rms_f); print(Kf); print("D =", Df.ravel(), " true:", Df_true)

In [ ]:
C.check_fisheye(Kf, Df, Kf_true, Df_true)

## 5. Undistort the fisheye: `balance` and the new camera matrix

In [ ]:
v = fviews[0]
fig, ax = plt.subplots(1, 3, figsize=(22, 5.5))
ax[0].imshow(v, cmap="gray"); ax[0].set_title("fisheye")
for a, bal in zip(ax[1:], (0.0, 1.0)):
    newK = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(Kf, Df, (1280, 960), np.eye(3), balance=bal)
    m1, m2 = cv2.fisheye.initUndistortRectifyMap(Kf, Df, np.eye(3), newK, (1280, 960), cv2.CV_16SC2)
    a.imshow(cv2.remap(v, m1, m2, cv2.INTER_LINEAR), cmap="gray"); a.set_title(f"balance = {bal}")
for a in ax: a.axis("off")
plt.show()

## 6. What if we use the wrong model?

Calibrate the fisheye views with the **standard** `cv2.calibrateCamera` (5 coefficients, then the rational model) and
compare the RMS values. Then undistort `fviews[0]` with each model. Where in the image do they disagree?

Hint: as long as every corner stays within roughly ±60° of the optical axis, a polynomial pinhole model can imitate a
fisheye lens. The difference appears near the image border, where only the fisheye model can still describe the lens.
Low RMS on the calibration views is not proof that the model is right everywhere.

In [ ]:
o32 = [o.reshape(-1, 3).astype(np.float32) for o in f_obj]; i32 = [i.reshape(-1, 1, 2).astype(np.float32) for i in f_img]
r5, *_ = cv2.calibrateCamera(o32, i32, (1280, 960), None, None)
r8, *_ = cv2.calibrateCamera(o32, i32, (1280, 960), None, None, flags=cv2.CALIB_RATIONAL_MODEL)
print(f"fisheye model RMS {rms_f:.3f} px | pinhole 5-coef RMS {r5:.3f} px | rational RMS {r8:.3f} px")

## Your turn

0. Run the fisheye calibration with `None, None` instead of `K0, np.zeros(4)` and `use_guess=False`. What happens?
1. Calibrate the ChArUco set with only 5, 10 and 20 views. Plot the fx error against the number of views.
2. Add noise: `v + np.random.normal(0, 8, v.shape)` (clip to 0–255) before detection. How do RMS and the K error change?
3. Make all ChArUco poses **frontal** (`tilt_deg=2` in `C.random_poses`). What happens to the focal-length error? Explain
   using Zhang's method (why do we need tilted views?).